# Task 11: ViT patch embedding + multi-head self-attention from scratch

In [3]:
import torch
import torch.nn as nn
from einops import rearrange
from einops.layers.torch import Rearrange


ModuleNotFoundError: No module named 'einops'

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_ch=3, dim=128):
        super().__init__()
        n_patches = (img_size//patch_size)**2
        self.proj = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch_size, p2=patch_size),
            nn.Linear(patch_size*patch_size*in_ch, dim)
        )
        self.cls_token = nn.Parameter(torch.randn(1,1,dim))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches+1, dim))

    def forward(self, x):
        b = x.shape[0]
        x = self.proj(x)
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1)
        return x + self.pos_embed


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        self.heads = heads
        self.scale = (dim//heads) ** -0.5
        self.qkv = nn.Linear(dim, dim*3)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b n (three h d) -> three b h n d', three=3, h=self.heads).unbind(0)
        attn = (q @ k.transpose(-1,-2)) * self.scale
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.out(out), attn


In [ ]:
patch_embed = PatchEmbed()
mha = MultiHeadAttention()
x = torch.randn(2, 3, 64, 64)
tokens = patch_embed(x)
out, attn = mha(tokens)
print(tokens.shape, out.shape, attn.shape)


In [ ]:
import matplotlib.pyplot as plt

plt.imshow(attn[0,0].detach().numpy())
plt.title("attention map, head 0")
plt.colorbar()
plt.show()
